In [1]:
import os
import pandas as pd
import numpy as np

# =========================
# PATHS
# =========================
PATH_EARLY_LOCAL = r"../../data/Riot_API/feature/datamart/early/dm_match_team_shap_local_early_with_meta.csv"
PATH_FULL_LOCAL  = r"../../data/Riot_API/feature/datamart/full/dm_match_team_shap_local_full_with_meta.csv"
# feature value source (wide)
PATH_EARLY_WIDE = r"../../data/Riot_API/feature/datamart/dm_match_team_early.csv"
PATH_FULL_WIDE  = r"../../data/Riot_API/feature/datamart/dm_match_team_feature.csv"

OUT_DIR = r"../../data/Riot_API/feature/tableau/TAB4"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_CASES = os.path.join(OUT_DIR, "dm_tab4_recovery_cases.csv")
OUT_LOCAL_LONG = os.path.join(OUT_DIR, "dm_tab4_recovery_case_local_long.csv")

# =========================
# CONFIG (case thresholds)
# =========================
EARLY_UNDERDOG_TH = 0.45   # early 불리 기준
EARLY_FAVOR_TH    = 0.55   # early 유리 기준
FULL_WIN_TH       = 0.50   # full 승리 기준

TOP_K_LOCAL = 15           # (match_id, team_id, model_name) 당 Local SHAP row 수
KEEP_TOPK_LONG = 15        # local_long에 남길 topK (보통 15 그대로)

# =========================
# LOAD
# =========================
early_long = pd.read_csv(PATH_EARLY_LOCAL)
full_long  = pd.read_csv(PATH_FULL_LOCAL)

print("early_long:", early_long.shape)
print("full_long :", full_long.shape)

# =========================
# 1) long -> team-level collapse (1 row per match_id, team_id, model_name)
# =========================
def collapse_local(long_df: pd.DataFrame, prob_col: str) -> pd.DataFrame:
    need = {"model_name","match_id","team_id",prob_col,"win_team","platform","queue_id","game_version","game_creation"}
    missing = need - set(long_df.columns)
    if missing:
        raise ValueError(f"Missing columns in local dm: {missing}")

    vc = long_df.groupby(["model_name","match_id","team_id"]).size().value_counts().head(5)
    print("rows-per-team value_counts:", vc.to_dict())

    meta_cols = ["platform","queue_id","game_version","game_creation","win_team"]
    out = (long_df
           .sort_values(["model_name","match_id","team_id"])
           .groupby(["model_name","match_id","team_id"], as_index=False)
           .agg({prob_col:"first", **{c:"first" for c in meta_cols}}))
    return out

early_team = collapse_local(early_long, "pred_prob_early")
full_team  = collapse_local(full_long,  "pred_prob_full")

print("early_team:", early_team.shape)
print("full_team :", full_team.shape)

# =========================
# 2) join early + full by (model_name, match_id, team_id)
# =========================
cases = early_team.merge(
    full_team,
    on=["model_name","match_id","team_id","platform","queue_id","game_version","game_creation","win_team"],
    how="inner"
)

print("joined cases:", cases.shape)

# =========================
# 3) derive labels / scores
# =========================
cases["early_state"] = np.select(
    [cases["pred_prob_early"] <= EARLY_UNDERDOG_TH,
     cases["pred_prob_early"] >= EARLY_FAVOR_TH],
    ["UNDERDOG","FAVOR"],
    default="NEUTRAL"
)

cases["full_state"] = np.where(cases["pred_prob_full"] >= FULL_WIN_TH, "WINPRED", "LOSEPRED")

cases["case_type"] = "NEUTRAL"
cases.loc[(cases["early_state"]=="UNDERDOG") & (cases["full_state"]=="WINPRED") & (cases["win_team"]==1), "case_type"] = "COMEBACK"
cases.loc[(cases["early_state"]=="FAVOR")    & (cases["full_state"]=="LOSEPRED") & (cases["win_team"]==0), "case_type"] = "THROW"
cases.loc[(cases["early_state"]=="FAVOR")    & (cases["full_state"]=="WINPRED") & (cases["win_team"]==1), "case_type"] = "STOMP_WIN"
cases.loc[(cases["early_state"]=="UNDERDOG") & (cases["full_state"]=="LOSEPRED") & (cases["win_team"]==0), "case_type"] = "STOMP_LOSS"

cases["delta_prob"] = cases["pred_prob_full"] - cases["pred_prob_early"]
cases["abs_delta_prob"] = cases["delta_prob"].abs()

# =========================
# 4) derive meta_window / queue_type / region (Tableau 필터용)
# - 현재 패치 체계: 15.x = 2025, 16.1~16.2 = 2026
# =========================
def meta_window(v):
    try:
        major = int(str(v).split(".")[0])
        if major == 15:
            return "2025"
        elif major == 16:
            return "2026"
        else:
            return "OTHER"
    except:
        return "OTHER"

cases["meta_window"] = cases["game_version"].apply(meta_window)
cases["queue_type"] = cases["queue_id"].map({420:"Solo", 440:"Flex"}).fillna("Other")

REGION_MAP = {
    # AMERICAS
    "BR1":"AMERICAS","LA1":"AMERICAS","LA2":"AMERICAS","NA1":"AMERICAS","OC1":"AMERICAS",
    # EUROPE
    "EUN1":"EUROPE","EUW1":"EUROPE","RU":"EUROPE","TR1":"EUROPE",
    # ASIA
    "KR":"ASIA","JP1":"ASIA","TW2":"ASIA",
    # SEA
    "SG2":"SEA","VN2":"SEA","ME1":"SEA"
}
cases["platform_u"] = cases["platform"].astype(str).str.upper()
cases["region"] = cases["platform_u"].map(REGION_MAP).fillna("UNKNOWN")

# =========================
# 5) save cases
# =========================
keep_cols = [
    "model_name","match_id","team_id",
    "platform_u","region","queue_id","queue_type","game_version","meta_window","game_creation",
    "win_team",
    "pred_prob_early","pred_prob_full","delta_prob","abs_delta_prob",
    "early_state","full_state","case_type"
]
cases_out = cases[keep_cols].rename(columns={"platform_u":"platform"}).copy()

cases_out.to_csv(OUT_CASES, index=False)
print("Saved:", OUT_CASES, cases_out.shape)
print(cases_out["case_type"].value_counts())

# =========================
# 6) build local_long (Early + Full local SHAP, joined with case labels)
# =========================
# 필요한 컬럼만 남기고 phase를 붙여서 concat
def prep_local_long(long_df, phase, prob_col):
    need = {"model_name","match_id","team_id",prob_col,"feature_name",
            "shap_logit","abs_shap_logit","delta_p","abs_delta_p","rank_in_row"}
    missing = need - set(long_df.columns)
    if missing:
        raise ValueError(f"[{phase}] Missing columns in local dm: {missing}")

    out = long_df.copy()
    out["phase"] = phase
    out["pred_prob"] = out[prob_col]

    # ---- feature_group / metric_group 보정 ----
    # feature_group: Early는 포지션 슬롯이 아니므로 상수로 둬도 됨
    if "feature_group" not in out.columns:
        out["feature_group"] = "EARLY" if phase == "EARLY" else "UNKNOWN"

    # metric_group: Early는 early_group을 metric_group으로 사용
    if "metric_group" not in out.columns:
        if phase == "EARLY":
            if "early_group" in out.columns:
                out["metric_group"] = out["early_group"]
            else:
                # early_group이 없는 경우: feature_name 규칙으로 최소 매핑
                def map_early_group(fn: str) -> str:
                    s = str(fn).lower()
                    if "gold" in s or "xp" in s or "cs" in s or "level" in s:
                        return "EARLY_ECON"
                    if "dmg" in s or "cc" in s or "champion_kill" in s:
                        return "EARLY_COMBAT"
                    if "ward" in s or "vision" in s:
                        return "EARLY_VISION"
                    if "dragon" in s or "horde" in s or "plate" in s or "elite" in s:
                        return "EARLY_OBJECTIVE"
                    return "EARLY_OTHER"
                out["metric_group"] = out["feature_name"].map(map_early_group)
        else:
            out["metric_group"] = "UNKNOWN"

    keep = ["model_name","match_id","team_id","phase","pred_prob",
            "feature_name","feature_group","metric_group",
            "shap_logit","abs_shap_logit","delta_p","abs_delta_p","rank_in_row"]
    return out[keep]

early_ll = prep_local_long(early_long, "EARLY", "pred_prob_early")
full_ll  = prep_local_long(full_long,  "FULL",  "pred_prob_full")

# topK clamp (혹시 TOP_K_LOCAL이 달라져도 안전)
early_ll = early_ll[early_ll["rank_in_row"] <= KEEP_TOPK_LONG].copy()
full_ll  = full_ll[full_ll["rank_in_row"] <= KEEP_TOPK_LONG].copy()

def attach_feature_values(local_long: pd.DataFrame,
                          wide_df: pd.DataFrame,
                          phase: str,
                          key_cols=("match_id","team_id"),
                          feature_col="feature_name",
                          out_col="feature_value") -> pd.DataFrame:
    """
    local_long (long): match_id, team_id, feature_name ... 에 대해,
    wide_df (wide): match_id, team_id, [feature columns...] 에서 실제 값을 붙인다.

    - 메모리/속도 위해 local_long에 실제로 등장하는 feature만 melt한다.
    """
    df = local_long.copy()

    # 필요한 feature 목록 (해당 phase에서만)
    need_features = sorted(df[feature_col].dropna().unique().tolist())

    # wide_df에 존재하는 feature만 가져오기 (불일치 안전)
    wide_cols = set(wide_df.columns)
    hit = [f for f in need_features if f in wide_cols]
    miss = [f for f in need_features if f not in wide_cols]

    print(f"[{phase}] need_features={len(need_features)}, hit={len(hit)}, miss={len(miss)}")
    if len(miss) > 0:
        print(f"[{phase}] (sample missing) {miss[:10]}")

    # melt할 최소 컬럼만 구성
    base_cols = [c for c in key_cols if c in wide_df.columns]
    if len(base_cols) != len(key_cols):
        raise ValueError(f"[{phase}] wide_df missing key cols: {set(key_cols)-set(base_cols)}")

    tmp = wide_df[base_cols + hit].copy()

    # melt -> (match_id, team_id, feature_name, feature_value)
    long_val = tmp.melt(id_vars=base_cols, value_vars=hit,
                        var_name=feature_col, value_name=out_col)

    # join back
    df = df.merge(long_val, on=list(key_cols)+[feature_col], how="left")

    # miss feature들은 NaN 유지(원하면 0으로도 가능하지만 추천은 NaN)
    return df

# ---- load wide DMs ----
early_wide = pd.read_csv(PATH_EARLY_WIDE)
full_wide  = pd.read_csv(PATH_FULL_WIDE)

# (권장) 키 컬럼 타입 통일
for df_ in [early_wide, full_wide]:
    df_["match_id"] = df_["match_id"].astype(str)
    df_["team_id"]  = df_["team_id"].astype(int)

early_ll["match_id"] = early_ll["match_id"].astype(str)
early_ll["team_id"]  = early_ll["team_id"].astype(int)
full_ll["match_id"]  = full_ll["match_id"].astype(str)
full_ll["team_id"]   = full_ll["team_id"].astype(int)

# ---- attach feature_value ----
early_ll = attach_feature_values(early_ll, early_wide, phase="EARLY",
                                 key_cols=("match_id","team_id"),
                                 feature_col="feature_name",
                                 out_col="feature_value")

full_ll  = attach_feature_values(full_ll, full_wide, phase="FULL",
                                 key_cols=("match_id","team_id"),
                                 feature_col="feature_name",
                                 out_col="feature_value")

# ---- concat final ----
local_long = pd.concat([early_ll, full_ll], ignore_index=True)

print("local_long:", local_long.shape)
print("feature_value null rate:", local_long["feature_value"].isna().mean())

# 케이스 하나 샘플 확인
sample = (local_long
          .sort_values(["abs_delta_p"], ascending=False)
          .head(10)[["phase","feature_name","feature_value","delta_p","abs_delta_p"]])
print(sample)

# case labels join
local_long = local_long.merge(
    cases_out[["model_name","match_id","team_id","platform","region","queue_id","queue_type","game_version","meta_window","game_creation",
               "win_team","pred_prob_early","pred_prob_full","delta_prob","abs_delta_prob","case_type"]],
    on=["model_name","match_id","team_id"],
    how="inner"
)

# 정렬(케이스 리스트 → 상세 드릴다운)
local_long = local_long.sort_values(["model_name","abs_delta_prob","match_id","team_id","phase","rank_in_row"], ascending=[True, False, True, True, True, True])

local_long.to_csv(OUT_LOCAL_LONG, index=False)
print("Saved:", OUT_LOCAL_LONG, local_long.shape)

# sanity: (match,team,phase)당 rows=KEEP_TOPK_LONG 인지
vc = local_long.groupby(["model_name","match_id","team_id","phase"]).size().value_counts().head(5)
print("rows per (match,team,phase) value_counts:", vc.to_dict())


early_long: (650520, 21)
full_long : (650520, 22)
rows-per-team value_counts: {15: 43368}
rows-per-team value_counts: {15: 43368}
early_team: (43368, 9)
full_team : (43368, 9)
joined cases: (8564, 10)
Saved: ../../data/Riot_API/feature/tableau/TAB4\dm_tab4_recovery_cases.csv (8564, 18)
case_type
STOMP_WIN     2496
STOMP_LOSS    2490
NEUTRAL       1558
THROW         1013
COMEBACK      1007
Name: count, dtype: int64
[EARLY] need_features=15, hit=15, miss=0
[FULL] need_features=44, hit=44, miss=0
local_long: (1301040, 14)
feature_value null rate: 0.027953022197626513
       phase                feature_name  feature_value   delta_p  abs_delta_p
794085  FULL  pos_SUP__gold_earned__diff        -9773.0 -0.463859     0.463859
803190  FULL  pos_SUP__gold_earned__diff        -8874.0 -0.459123     0.459123
773835  FULL  pos_SUP__gold_earned__diff        -9746.0 -0.458355     0.458355
795000  FULL  pos_SUP__gold_earned__diff        -8941.0 -0.458035     0.458035
687225  FULL  pos_SUP__gold_earned